# Alpaca SDK Toy Example

This notebook uses Alpaca's Python SDK directly against paper trading credentials. It is intentionally read-first and tiny.

It demonstrates:

- loading credentials from environment variables
- creating a paper `TradingClient`
- reading account and market clock state
- fetching a small batch of recent bars

It does not submit orders.

In [1]:
import os

ALPACA_API_KEY = os.getenv("ALPACA_API_KEY")
ALPACA_SECRET_KEY = os.getenv("ALPACA_SECRET_KEY")

if not ALPACA_API_KEY or not ALPACA_SECRET_KEY:
    raise RuntimeError("Set ALPACA_API_KEY and ALPACA_SECRET_KEY in .env, then restart the Jupyter container.")

print("credentials loaded")

credentials loaded


In [2]:
from alpaca.trading.client import TradingClient

trading_client = TradingClient(ALPACA_API_KEY, ALPACA_SECRET_KEY, paper=True)

account = trading_client.get_account()
{
    "status": account.status,
    "cash": str(account.cash),
    "equity": str(account.equity),
    "buying_power": str(account.buying_power),
}

{'status': <AccountStatus.ACTIVE: 'ACTIVE'>,
 'cash': '100000',
 'equity': '100000',
 'buying_power': '200000'}

In [3]:
clock = trading_client.get_clock()
{
    "is_open": clock.is_open,
    "timestamp": clock.timestamp.isoformat(),
    "next_open": clock.next_open.isoformat(),
    "next_close": clock.next_close.isoformat(),
}

{'is_open': False,
 'timestamp': '2026-05-09T15:46:52.108557-04:00',
 'next_open': '2026-05-11T09:30:00-04:00',
 'next_close': '2026-05-11T16:00:00-04:00'}

In [4]:
from datetime import UTC, datetime, timedelta

from alpaca.data.enums import DataFeed
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

data_client = StockHistoricalDataClient(ALPACA_API_KEY, ALPACA_SECRET_KEY)

symbol = "SPY"
end = datetime.now(UTC)
start = end - timedelta(days=14)

request = StockBarsRequest(
    symbol_or_symbols=[symbol],
    timeframe=TimeFrame.Day,
    start=start,
    end=end,
    limit=5,
    feed=DataFeed.IEX,
)

bars_by_symbol = data_client.get_stock_bars(request).data
bars = bars_by_symbol.get(symbol, [])

[
    {
        "timestamp": bar.timestamp.isoformat(),
        "open": float(bar.open),
        "high": float(bar.high),
        "low": float(bar.low),
        "close": float(bar.close),
        "volume": int(bar.volume),
    }
    for bar in bars
]

[{'timestamp': '2026-04-27T04:00:00+00:00',
  'open': 713.18,
  'high': 715.59,
  'low': 712.33,
  'close': 715.165,
  'volume': 1329665},
 {'timestamp': '2026-04-28T04:00:00+00:00',
  'open': 711.75,
  'high': 712.88,
  'low': 709.27,
  'close': 711.68,
  'volume': 1225364},
 {'timestamp': '2026-04-29T04:00:00+00:00',
  'open': 710.98,
  'high': 712.19,
  'low': 708.435,
  'close': 711.59,
  'volume': 889917},
 {'timestamp': '2026-04-30T04:00:00+00:00',
  'open': 714.67,
  'high': 719.71,
  'low': 710.465,
  'close': 718.41,
  'volume': 1976926},
 {'timestamp': '2026-05-01T04:00:00+00:00',
  'open': 721.245,
  'high': 724.85,
  'low': 720.485,
  'close': 720.49,
  'volume': 1437357}]

This is what the orchestrator automates: fetch market/account state, compute features, call inference, apply policy, and persist the result.